# View LSSTCam DeepCoadd And Objects in Firefly

Display deep-coadd patches around a selected DDF sky field, overlaid with
photometrically stable stars from `targets_matched_lsst_objects.csv`.
Markers are annotated with the SIMBAD name and spectral type.

**Workflow:**
1. Connect to the Butler repository and load the skymap.
2. Choose a target DDF field in the **Configuration** section.
3. Load the stable-star catalogue and filter by the selected field.
4. **Analyse the patch coverage** per DDF: how many distinct patches cover
   the stars? Apply the `npatchmax` decision rule:
   - `n_patches > npatchmax` → load only the required patches in `BANDSEL`.
   - `n_patches ≤ npatchmax` → load those patches for all 6 bands.
5. Display each coadd in Firefly with markers at the star positions,
   annotated with SIMBAD name + spectral type.

- **Author:** Sylvie Dagoret-Campagne
- **Created:** 2026-06-28
- **Last update:** 2026-06-28
- **LSST pipelines:** w_2026_10

### Info on Firefly
- Code : https://github.com/Caltech-IPAC/firefly/tree/dev
- API : https://caltech-ipac.github.io/firefly_client/
- tutorial here : https://exoplanetarchive.ipac.caltech.edu/firefly/onlinehelp/firefly/visualization.html#imageinfo

## 1. Imports

In [ ]:
import gc
import logging
import os
import sys

import numpy as np
import pandas as pd

import matplotlib
import matplotlib.pyplot as plt

from lsst.daf.butler import Butler
import lsst.geom as geom
from lsst.geom import SpherePoint, degrees
import lsst.afw.display as afwDisplay
import firefly_client.plot as ffplt

from lsst.skymap import Index2D

In [ ]:
def dataset_type_exists(butler, name):
    try:
        butler.registry.getDatasetType(name)
        return True
    except KeyError:
        return False

## 2. Logging

In [ ]:
log = logging.getLogger()
log.setLevel(logging.INFO)
if not log.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setLevel(logging.INFO)
    formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
    handler.setFormatter(formatter)
    log.addHandler(handler)
log.info("Logging configured.")

## 3. Configuration

**Edit only this cell** to select the target field, the Butler collection,
and the fallback band. Everything else runs automatically.


In [ ]:
# ── Notebook tag and I/O directories ─────────────────────────────────────────
NB_TAG = "DEEPCOADDSOBJ_03"
DIR_DATA_IN = f"data_{NB_TAG}_in"
DIR_DATA_OUT = f"data_{NB_TAG}_out"
DIR_FIGS = f"figs_{NB_TAG}"
os.makedirs(DIR_DATA_OUT, exist_ok=True)
os.makedirs(DIR_FIGS, exist_ok=True)
log.info(f"Data input : {os.path.abspath(DIR_DATA_IN)}")
log.info(f"Data output: {os.path.abspath(DIR_DATA_OUT)}")
log.info(f"Figs       : {os.path.abspath(DIR_FIGS)}")

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)


def savefig(name: str) -> None:
    """Save the current figure to both PDF and PNG inside DIR_FIGS."""
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    log.info(f"  -> saved {name}.{{pdf,png}}")


log.info("Directory configuration done.")

In [ ]:
# ============================================================
# USER CONFIGURATION — edit only this cell to change the target
# ============================================================

# --- Butler repository and collections ---
repo = "dp2_prep"
collection = [
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage1",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage2",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage3",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage4",
]
instrument = "LSSTCam"
skymapName = "lsst_cells_v2"

# --- Field selection ---
# lsstcam_fields keys: "COSMOS", "XMM-LSS", "ELAIS-S1", "ECDFS", "EDFS_a", "EDFS_b"
key = "COSMOS"  # <-- change to select a DDF (or ECDFS, no other coadd 2026-06-28)

# --- Band logic ---
BANDSEL = "r"  # band used when n_patches > npatchmax
npatchmax = 2  # threshold: if #patches > npatchmax, show only BANDSEL

ALL_BANDS = ["u", "g", "r", "i", "z", "y"]

# --- Firefly marker style ---
MARKER_SIZE = 50  # circle radius in pixels
MARKER_COLOR = "red"  # colour of the overlay circles

## 4. DDF field definitions

In [ ]:
lsstcam_fields = {}
lsstcam_fields["COSMOS"] = {"field_name": "COSMOS Deep Drilling Field", "ra": 150.11, "dec": 2.23}
lsstcam_fields["XMM-LSS"] = {"field_name": "XMM-LSS Deep Drilling Field", "ra": 35.57, "dec": -4.82}
lsstcam_fields["ELAIS-S1"] = {"field_name": "ELAIS-S1 Deep Drilling Field", "ra": 9.45, "dec": -44.02}
lsstcam_fields["ECDFS"] = {"field_name": "Extended Chandra Deep Field South", "ra": 52.98, "dec": -28.12}
lsstcam_fields["EDFS_a"] = {"field_name": "Euclid Deep Field South (pointing a)", "ra": 58.90, "dec": -49.32}
lsstcam_fields["EDFS_b"] = {"field_name": "Euclid Deep Field South (pointing b)", "ra": 63.60, "dec": -47.60}

In [ ]:
the_target = lsstcam_fields[key]
target_ra = the_target["ra"]
target_dec = the_target["dec"]
target_point = SpherePoint(target_ra, target_dec, degrees)
log.info(f"Selected field : {key} — {the_target['field_name']}")
log.info(f"  (RA, Dec) = ({target_ra:.3f}, {target_dec:.3f}) deg")

## 5. Initialise the Butler

In [ ]:
butler = Butler(repo, collections=collection)
registry = butler.registry
log.info(f"Butler initialised | repo: {repo}")

In [ ]:
skymap = butler.get("skyMap", skymap=skymapName, collections=collection)

## 6. Auto-discover the deepCoadd dataset type

In [ ]:
COADD_CANDIDATES = [
    "legacy_deep_coadd",
    "deep_coadd",
    "deepCoadd_calexp",
    "deepCoadd",
    "goodSeeingCoadd",
]
all_coadd_types = [d.name for d in butler.registry.queryDatasetTypes() if "coadd" in d.name.lower()]
print("Coadd dataset types in registry:", all_coadd_types)

COADD_DATASET = None
for name in COADD_CANDIDATES:
    if dataset_type_exists(butler, name):
        COADD_DATASET = name
        break

if COADD_DATASET is None:
    raise RuntimeError(f"No recognised deepCoadd dataset type found. Available: {all_coadd_types}")
log.info(f"✔ Using deepCoadd dataset type: '{COADD_DATASET}'")

## 7. Load the stable-star catalogue

Generated by `04_FindObjects/.../02_MatchTargetsWithLSSTCamObjects.ipynb`.


In [ ]:
filename_input = "targets_matched_lsst_objects.csv"
fullfilename_input = os.path.join(DIR_DATA_IN, filename_input)
df_all = pd.read_csv(fullfilename_input)
log.info(f"Total rows in catalogue: {len(df_all)}")
log.info(f"Available fields: {df_all['field'].unique()}")

## 8. Patch-coverage analysis — per DDF

For each DDF present in the catalogue we count:
- the distinct `(lsst_tract, lsst_patch)` pairs that contain at least one stable star
- whether that count exceeds `npatchmax`

This drives the loading strategy further below.


In [ ]:
log.info("=" * 70)
log.info(f"PATCH-COVERAGE ANALYSIS  (npatchmax = {npatchmax})")
log.info("=" * 70)

summary_rows = []
for ddf_name, grp in df_all.groupby("field"):
    patches_in_ddf = grp[["lsst_tract", "lsst_patch"]].drop_duplicates()
    n_patch = len(patches_in_ddf)
    n_stars = len(grp)
    strategy = "single band (BANDSEL)" if n_patch > npatchmax else "all 6 bands"
    log.info(f"  {ddf_name:<12}  stars={n_stars:3d}  distinct_patches={n_patch:3d}  → {strategy}")
    for _, row in patches_in_ddf.iterrows():
        log.info(f"               tract={int(row['lsst_tract'])}  patch={int(row['lsst_patch'])}")
    summary_rows.append({"field": ddf_name, "n_stars": n_stars, "n_patches": n_patch, "strategy": strategy})

df_summary = pd.DataFrame(summary_rows).set_index("field")
log.info("\nSummary table:")
log.info(df_summary.to_string())

## 9. Select the target DDF and apply the loading strategy

In [ ]:
# Filter catalogue to the selected field
df_field = df_all[df_all["field"] == key].copy().reset_index(drop=True)
log.info(f"Stars in {key}: {len(df_field)}")

# Distinct (tract, patch) pairs for this field
patch_pairs = df_field[["lsst_tract", "lsst_patch"]].drop_duplicates().reset_index(drop=True)
n_patches_field = len(patch_pairs)
log.info(f"Distinct patches: {n_patches_field}")
log.info(patch_pairs.to_string())

# Decide on the band list
if n_patches_field > npatchmax:
    bands_to_load = [BANDSEL]
    log.info(f"\n→ n_patches ({n_patches_field}) > npatchmax ({npatchmax}): loading band '{BANDSEL}' only.")
else:
    bands_to_load = ALL_BANDS
    log.info(f"\n→ n_patches ({n_patches_field}) ≤ npatchmax ({npatchmax}): loading all 6 bands.")

## 10. Resolve tract info from the skymap

In [ ]:
tract_info = skymap.findTract(target_point)
tractNbSel = tract_info.getId()
patch_info = tract_info.findPatch(target_point)
patchNbSel = patch_info.getSequentialIndex()

log.info(f"Central tract : {tractNbSel}")
log.info(f"Central patch : {patchNbSel}  index={patch_info.getIndex()}")

## 11. Helper: RA,Dec → pixel coordinates via the coadd WCS

The `afw_display.dot()` API works in **pixel** coordinates of the
currently displayed image.  We use `wcs.skyToPixel()` to convert.


In [ ]:
def radec_to_pixel(wcs, ra_deg, dec_deg):
    """
    Convert (RA, Dec) in degrees to (x, y) pixel coordinates
    using an lsst.afw.geom.SkyWcs.

    Returns
    -------
    (x, y) : float tuple  —  pixel coordinates (0-based, FITS convention)
    """
    sky_point = SpherePoint(ra_deg * degrees, dec_deg * degrees)
    pix_point = wcs.skyToPixel(sky_point)
    return pix_point.getX(), pix_point.getY()


def build_label(simbad_id, spectral_type):
    """
    Build a short annotation string: 'NAME (SpType)' if SpType exists.
    """
    sid = str(simbad_id).strip() if pd.notna(simbad_id) else ""
    spt = str(spectral_type).strip() if pd.notna(spectral_type) else ""
    if spt:
        return f"{sid} ({spt})"
    return sid

## 12. Initialise Firefly display

In [ ]:
afwDisplay.setDefaultBackend("firefly")

## 13. Load coadds and display with star overlays

For each `(tract, patch)` pair that contains at least one stable star,
and for each band in `bands_to_load`:

1. Fetch the deepCoadd from the Butler.
2. Display it in a new Firefly frame.
3. Overlay circles at the pixel positions of the stars, annotated with
   their SIMBAD name and spectral type.


In [ ]:
frame_counter = 0  # Firefly frame index (1-based)

for _, pair in patch_pairs.iterrows():
    tract_nb = int(pair["lsst_tract"])
    patch_nb = int(pair["lsst_patch"])

    # Stars that fall in this (tract, patch)
    mask_patch = (df_field["lsst_tract"] == tract_nb) & (df_field["lsst_patch"] == patch_nb)
    df_patch = df_field[mask_patch].copy()
    n_stars_patch = len(df_patch)

    for band in bands_to_load:
        frame_counter += 1

        title = f"{key} | band={band} | tract={tract_nb} patch={patch_nb} | {n_stars_patch} stars"
        print(f"Frame {frame_counter}: {title}")

        dataId = {
            "band": band,
            "tract": tract_nb,
            "patch": patch_nb,
            "skymap": skymapName,
        }

        try:
            coadd_exp = butler.get(COADD_DATASET, dataId)
        except Exception as exc:
            print(f"  ✗ Butler.get failed for {dataId}: {exc}")
            frame_counter -= 1  # don't waste a frame number
            continue

        wcs = coadd_exp.getWcs()

        # Display the coadd
        display = afwDisplay.Display(frame=frame_counter)
        display.scale("asinh", "zscale")
        display.setMaskTransparency(100)
        display.mtv(coadd_exp.image, title=title)

        # Overlay stable-star markers
        with display.Buffering():
            for _, star in df_patch.iterrows():
                ra_star = star["lsst-obj_coord_ra"]
                dec_star = star["lsst-obj_coord_dec"]

                # Skip stars with missing coordinates
                if pd.isna(ra_star) or pd.isna(dec_star):
                    continue

                try:
                    px, py = radec_to_pixel(wcs, ra_star, dec_star)
                except Exception:
                    continue  # star outside image footprint

                label = build_label(star.get("simbad_id"), star.get("spectral_type"))
                log.info(f"\t ==> - star : {label} on frame {frame_counter} patch {patch_nb}")

                # Circle marker
                display.dot("o", px, py, size=MARKER_SIZE, ctype=MARKER_COLOR)
                # Text annotation just above the circle
                if label:
                    display.dot("+", px, py + MARKER_SIZE + 5, size=1, ctype=MARKER_COLOR)
                    # Use afw_display label if available
                    try:
                        display.label(px, py + MARKER_SIZE + 20, label, ctype=MARKER_COLOR)
                        # display.text(px, py + MARKER_SIZE + 20, label, ctype=MARKER_COLOR)
                    except AttributeError:
                        pass  # label() not available in all backend versions

        log.info(f"  ✔ {n_stars_patch} markers drawn")

        del coadd_exp, wcs
        gc.collect()

log.info(f"\nTotal frames displayed: {frame_counter}")

## 14. (Optional) Matplotlib preview with star overlay

Quick sanity-check: display the first available patch in matplotlib
and overplot the stars to verify the WCS conversion.


In [ ]:
from astropy.visualization import ZScaleInterval
from astropy.wcs import WCS as AstropyWCS
import astropy.units as u

first_pair = patch_pairs.iloc[0]
tract_nb0 = int(first_pair["lsst_tract"])
patch_nb0 = int(first_pair["lsst_patch"])
band_preview = bands_to_load[0]

dataId0 = {"band": band_preview, "tract": tract_nb0, "patch": patch_nb0, "skymap": skymapName}

coadd0 = butler.get(COADD_DATASET, dataId0)
img_arr0 = coadd0.image.array
wcs0 = coadd0.getWcs()

# Build astropy WCS for matplotlib projection
header0 = wcs0.getFitsMetadata().toDict()
wcs_ap0 = AstropyWCS(header0)

interval = ZScaleInterval()
vmin0, vmax0 = interval.get_limits(img_arr0)

# Stars for this patch
mask0 = (df_field["lsst_tract"] == tract_nb0) & (df_field["lsst_patch"] == patch_nb0)
df_patch0 = df_field[mask0]

fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection=wcs_ap0)

ax.imshow(img_arr0, origin="lower", cmap="gray", vmin=vmin0, vmax=vmax0)

# Overlay star positions
for _, star in df_patch0.iterrows():
    ra_s = star["lsst-obj_coord_ra"]
    dec_s = star["lsst-obj_coord_dec"]
    if pd.isna(ra_s) or pd.isna(dec_s):
        continue
    try:
        px, py = radec_to_pixel(wcs0, ra_s, dec_s)
    except Exception:
        continue
    label = build_label(star.get("simbad_id"), star.get("spectral_type"))
    ax.plot(px, py, "o", mfc="none", mec=MARKER_COLOR, ms=MARKER_SIZE // 2, mew=1.5)
    if label:
        ax.annotate(
            label,
            xy=(px, py),
            xytext=(6, 6),
            textcoords="offset points",
            color=MARKER_COLOR,
            fontsize=7,
            arrowprops=dict(arrowstyle="-", color=MARKER_COLOR, lw=0.5),
        )

ax.coords.grid(True, color="white", ls="dotted", alpha=0.4)
ax.coords[0].set_axislabel("RA")
ax.coords[1].set_axislabel("Dec")
ax.invert_xaxis()

title_mpl = f"{the_target['field_name']} | band={band_preview} " f"| tract={tract_nb0} patch={patch_nb0}"
ax.set_title(title_mpl, pad=20)

plt.colorbar(ax.get_images()[0], ax=ax, pad=0.15, shrink=0.82, aspect=30).set_label("Flux")

savefig(f"{key}_band{band_preview}_tract{tract_nb0}_patch{patch_nb0}_stars")
plt.show()

del coadd0, img_arr0, wcs0
gc.collect()

## 15. Summary

Print the complete loading plan that was applied for the selected DDF.


In [ ]:
log.info("=" * 60)
log.info(f"DDF            : {key} — {the_target['field_name']}")
log.info(f"Stars loaded   : {len(df_field)}")
log.info(f"Distinct patches: {n_patches_field}")
log.info(f"npatchmax      : {npatchmax}")
log.info(f"Strategy       : bands = {bands_to_load}")
log.info(f"Frames in Firefly: {frame_counter}")
log.info("=" * 60)
log.info(patch_pairs.to_string())

In [ ]:
# display.clearViewer()